# Notebook 06 — Final RQ1 Evaluation: Cascaded C0 vs Direct D0

**Purpose:** final paired evaluation on the frozen `rq1_test` set. This notebook does **not train or tune** any model.

Stages:

`verify → unlock_test → run_cascaded → run_direct → finalize`

Scientific boundary:
- `verify` reads only upstream summaries/contracts; it must not open `rq1_test.csv`.
- `unlock_test` is the first legitimate full-content read of the frozen test and requires an explicit flag.
- after unlock, NB03/NB04/NB05 are treated as immutable upstream artifacts.
- C0 and D0 must produce predictions for the exact same ordered UID list.


In [ ]:
# Cell 1 — OPERATOR CONTROLS ONLY (excluded from scientific source fingerprint)
# Change only these values between stages.
RQ1_STAGE = "verify"  # verify | unlock_test | run_cascaded | run_direct | finalize
ALLOW_FROZEN_TEST_ACCESS = False  # True only from unlock_test onward

# Fill exact completed contract-scoped state directories after NB05 finishes.
ASR_STATE_DIR = ""
MT_STATE_DIR = ""
DIRECT_STATE_DIR = ""

# Full inference requires CUDA; verify/unlock do not.
DEVICE = "cuda"


In [ ]:
# Cell 2 — Imports, locked config, runtime/source identity
from pathlib import Path
import sys
import json, math, time, gc
import pandas as pd
import torch, yaml

def _bootstrap_project_root(start: Path) -> Path:
    node = start.resolve()
    for cand in [node, *node.parents]:
        if (
            (cand / "requirements.txt").is_file()
            and (cand / "src").is_dir()
            and (cand / "data" / "manifests").is_dir()
        ):
            return cand
    raise RuntimeError(
        f"Cannot locate bahnar-s2tt-thesis root from {node} "
        "(need requirements.txt + src/ + data/manifests/)"
    )

PROJECT_ROOT = _bootstrap_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rq1_runtime_paths import find_bahnar_project_root, resolve_rq1_runtime_paths
from src.rq1_contract import (
    STATUS_RQ1_VERIFY, STATUS_RQ1_UNLOCK, STATUS_RQ1_C0, STATUS_RQ1_D0, STATUS_RQ1_FINAL,
    BOOTSTRAP_METHOD_PAIRED_CLUSTER, BOOTSTRAP_UNIT_GROUP_ID,
    compute_rq1_source_fingerprint, assert_locked_runtime, build_final_contract,
    assert_rq1_test_contract, assert_unlocked_final_contract_matches_current,
    assert_locked_manifest_bundle_matches_contract, assert_rq1_bootstrap_gate,
    assert_stage_summary_matches_final_contract, assert_prediction_uid_order,
    derive_final_status, finite_metric_dict, sha256_file, sha256_json,
    collect_runtime_provenance, atomic_write_json, atomic_write_csv,
)
from src.rq1_evaluation import (
    verify_upstream_handoffs, unlock_frozen_test_manifest, join_paired_predictions,
    system_metrics, paired_bootstrap, slice_metrics, error_examples, prediction_artifact_manifest,
    recompute_and_assert_finalize_integrity,
)
from src.rq1_audio import ensure_verified_frozen_audio
from src.rq1_restore import (
    restore_asr_for_rq1, restore_mt_for_rq1, restore_direct_for_rq1,
    verify_upstream_checkpoint_artifacts, assert_checkpoint_proof_matches_upstream,
    assert_restored_checkpoint_matches_proof, cleanup_local_rq1_experiment,
    run_cascaded_c0_peak_safe, run_direct_d0_peak_safe,
)
from src.rq1_inference import run_asr_predictions, run_mt_from_asr, run_direct_predictions

assert find_bahnar_project_root(Path.cwd()) == PROJECT_ROOT
CFG = yaml.safe_load((PROJECT_ROOT / "configs/rq1.yaml").read_text(encoding="utf-8"))
RUNTIME = resolve_rq1_runtime_paths(project_root=PROJECT_ROOT)
SOURCE_FP = compute_rq1_source_fingerprint(PROJECT_ROOT)
ACTUAL_RUNTIME = collect_runtime_provenance()
assert_locked_runtime(CFG["runtime"], ACTUAL_RUNTIME)
print("PROJECT_ROOT=", PROJECT_ROOT)
print("source_fingerprint=", SOURCE_FP["aggregate_sha256"])
print("runtime=", json.dumps(ACTUAL_RUNTIME, indent=2))


In [ ]:
# Cell 3 — Common fail-closed helpers
ALLOWED_STAGES = {"verify", "unlock_test", "run_cascaded", "run_direct", "finalize"}
if RQ1_STAGE not in ALLOWED_STAGES:
    raise RuntimeError(f"Unknown RQ1_STAGE={RQ1_STAGE!r}")

if not ASR_STATE_DIR or not MT_STATE_DIR or not DIRECT_STATE_DIR:
    raise RuntimeError("Set exact ASR_STATE_DIR, MT_STATE_DIR and DIRECT_STATE_DIR in Cell 1")

UPSTREAM = verify_upstream_handoffs(
    asr_state_dir=ASR_STATE_DIR,
    mt_state_dir=MT_STATE_DIR,
    direct_state_dir=DIRECT_STATE_DIR,
)
UPSTREAM_META_KEY = sha256_json({
    "asr": UPSTREAM["asr_handoff_hash"],
    "mt": UPSTREAM["mt_handoff_hash"],
    "direct": UPSTREAM["direct_handoff_hash"],
    "source": SOURCE_FP["aggregate_sha256"],
    "runtime": ACTUAL_RUNTIME,
})
VERIFY_DIR = RUNTIME.durable_state_root / f"verify_{UPSTREAM_META_KEY[:16]}"
VERIFY_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PROOF_PATH = VERIFY_DIR / "checkpoint_proof.json"
if RQ1_STAGE == "verify":
    # One-time CPU restore+load proof. unlock_test reuses this durable proof.
    CHECKPOINT_PROOF = verify_upstream_checkpoint_artifacts(
        asr_state_dir=ASR_STATE_DIR, mt_state_dir=MT_STATE_DIR, direct_state_dir=DIRECT_STATE_DIR,
        asr_local_ckpt_root=RUNTIME.local_asr_ckpt_root,
        mt_local_ckpt_root=RUNTIME.local_mt_ckpt_root,
        direct_local_ckpt_root=RUNTIME.local_direct_ckpt_root,
        cleanup_local_after_verify=True,
        load_models=True,
        device="cpu",
    )
    atomic_write_json(CHECKPOINT_PROOF_PATH, CHECKPOINT_PROOF)
else:
    if not CHECKPOINT_PROOF_PATH.is_file():
        raise RuntimeError("Missing checkpoint_proof.json; run verify first")
    CHECKPOINT_PROOF = json.loads(CHECKPOINT_PROOF_PATH.read_text(encoding="utf-8"))
    assert_checkpoint_proof_matches_upstream(CHECKPOINT_PROOF, UPSTREAM)

# Bind verified checkpoint identity into every upstream handoff hash before test unlock.
UPSTREAM["checkpoint_proof"] = {
    "asr": CHECKPOINT_PROOF.get("asr"),
    "mt": CHECKPOINT_PROOF.get("mt"),
    "direct": CHECKPOINT_PROOF.get("direct"),
    "proof_hash": CHECKPOINT_PROOF.get("proof_hash"),
}
VERIFY_SUMMARY = {
    "status": STATUS_RQ1_VERIFY,
    "upstream_meta_key": UPSTREAM_META_KEY,
    "asr_handoff_hash": UPSTREAM["asr_handoff_hash"],
    "mt_handoff_hash": UPSTREAM["mt_handoff_hash"],
    "direct_handoff_hash": UPSTREAM["direct_handoff_hash"],
    "checkpoint_proof_hash": CHECKPOINT_PROOF.get("proof_hash"),
    "source_fingerprint": SOURCE_FP["aggregate_sha256"],
    "runtime": ACTUAL_RUNTIME,
}
atomic_write_json(VERIFY_DIR / "verify_summary.json", VERIFY_SUMMARY)

def load_unlocked_context():
    if not ALLOW_FROZEN_TEST_ACCESS:
        raise RuntimeError("Post-unlock stages require ALLOW_FROZEN_TEST_ACCESS=True")
    latest = RUNTIME.durable_state_root / "LATEST_UNLOCKED.json"
    if not latest.is_file():
        raise RuntimeError("No LATEST_UNLOCKED.json; run unlock_test first")
    meta = json.loads(latest.read_text(encoding="utf-8"))
    state = Path(meta["state_dir"])
    tc = json.loads((state / "rq1_test_contract.json").read_text(encoding="utf-8"))
    fc = json.loads((state / "rq1_final_contract.json").read_text(encoding="utf-8"))
    assert_unlocked_final_contract_matches_current(
        fc,
        test_contract=tc,
        asr_handoff_hash=UPSTREAM["asr_handoff_hash"],
        mt_handoff_hash=UPSTREAM["mt_handoff_hash"],
        direct_handoff_hash=UPSTREAM["direct_handoff_hash"],
        source_fingerprint_sha256=SOURCE_FP["aggregate_sha256"],
        runtime_versions=ACTUAL_RUNTIME,
        seed=int(CFG["seed"]),
        bootstrap_samples=int(CFG["bootstrap_samples"]),
        confidence=float(CFG["confidence"]),
        checkpoint_proof=CHECKPOINT_PROOF,
        latest_meta=meta,
    )
    assert_locked_manifest_bundle_matches_contract(project_root=PROJECT_ROOT, test_contract=tc)
    test_df = pd.read_csv(PROJECT_ROOT / "data" / "manifests" / "rq1_test.csv")
    assert_rq1_test_contract(test_df, tc, manifest_path=PROJECT_ROOT / "data" / "manifests" / "rq1_test.csv")
    return state, test_df, tc, fc


In [ ]:
# Cell 4 — Stage: verify (NO frozen test read)
if RQ1_STAGE == "verify":
    if ALLOW_FROZEN_TEST_ACCESS:
        raise RuntimeError("verify must run with ALLOW_FROZEN_TEST_ACCESS=False")
    print(json.dumps(VERIFY_SUMMARY, indent=2))
    print("READY_FOR_UNLOCK_TEST = True")


In [ ]:
# Cell 5 — Stage: unlock_test (first legitimate full test read)
if RQ1_STAGE == "unlock_test":
    test_df, unlock = unlock_frozen_test_manifest(
        project_root=PROJECT_ROOT,
        upstream_handoff=UPSTREAM,
        allow_frozen_test_access=ALLOW_FROZEN_TEST_ACCESS,
        dataset_id=CFG["dataset"]["id"],
        dataset_revision=CFG["dataset"]["revision"],
        parquet_revision=CFG["dataset"]["parquet_revision"],
        expected_test_count=CFG["expected_test_count"],
    )
    TEST_CONTRACT = unlock["contract"]
    FINAL_CONTRACT = build_final_contract(
        test_contract=TEST_CONTRACT,
        asr_handoff_hash=UPSTREAM["asr_handoff_hash"],
        mt_handoff_hash=UPSTREAM["mt_handoff_hash"],
        direct_handoff_hash=UPSTREAM["direct_handoff_hash"],
        source_fingerprint_sha256=SOURCE_FP["aggregate_sha256"],
        runtime_versions=ACTUAL_RUNTIME,
        seed=CFG["seed"],
        bootstrap_samples=CFG["bootstrap_samples"],
        confidence=CFG["confidence"],
        bootstrap_method=BOOTSTRAP_METHOD_PAIRED_CLUSTER,
        bootstrap_unit=BOOTSTRAP_UNIT_GROUP_ID,
        cluster_col=BOOTSTRAP_UNIT_GROUP_ID,
        checkpoint_proof=CHECKPOINT_PROOF,
    )
    STATE_DIR = RUNTIME.state_dir(FINAL_CONTRACT["rq1_final_contract_hash"])
    STATE_DIR.mkdir(parents=True, exist_ok=True)
    atomic_write_json(STATE_DIR / "rq1_test_contract.json", TEST_CONTRACT)
    atomic_write_json(STATE_DIR / "rq1_final_contract.json", FINAL_CONTRACT)
    atomic_write_json(STATE_DIR / "checkpoint_proof.json", CHECKPOINT_PROOF)
    atomic_write_json(STATE_DIR / "rq1_unlock_summary.json", {
        "status": STATUS_RQ1_UNLOCK,
        "test_contract": TEST_CONTRACT,
        "final_contract_hash": FINAL_CONTRACT["rq1_final_contract_hash"],
        "overlap": unlock.get("overlap"),
        "checkpoint_proof_hash": CHECKPOINT_PROOF.get("proof_hash"),
        "parquet_revision": unlock.get("parquet_revision"),
        "parquet_revision_source": unlock.get("parquet_revision_source"),
        "n_test": int(len(test_df)),
        "n_clusters": int(TEST_CONTRACT.get("n_clusters") or 0),
    })
    atomic_write_json(RUNTIME.durable_state_root / "LATEST_UNLOCKED.json", {
        "state_dir": str(STATE_DIR),
        "final_contract_hash": FINAL_CONTRACT["rq1_final_contract_hash"],
        "source_fingerprint": SOURCE_FP["aggregate_sha256"],
    })
    print(json.dumps({
        "status": STATUS_RQ1_UNLOCK,
        "state_dir": str(STATE_DIR),
        "n_test": len(test_df),
        "n_clusters": TEST_CONTRACT.get("n_clusters"),
        "parquet_revision_source": unlock.get("parquet_revision_source"),
    }, indent=2))


In [ ]:
# Cell 6 — Shared post-unlock audio materialization + PCM integrity verify
# Executed only by C0/D0 stages. Every frozen row must materialize; no filtering/drop is allowed.

def ensure_frozen_audio(test_df, state_dir, *, parquet_revision: str):
    # Use the unlocked contract/config pin — never an arbitrary Hub HEAD revision.
    rep = ensure_verified_frozen_audio(
        test_df,
        state_dir=state_dir,
        dataset_id=CFG["dataset"]["id"],
        parquet_revision=str(parquet_revision),
        audio_dir=RUNTIME.local_audio_dir,
        parquet_cache_dir=RUNTIME.hf_parquet_cache_dir,
    )
    return rep["audio_paths"], rep["audio_integrity_sha256"]


In [ ]:
# Cell 7 — Stage: run_cascaded (C0), exception-safe GPU+disk peak control
if RQ1_STAGE == "run_cascaded":
    if DEVICE != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("run_cascaded requires CUDA")
    STATE_DIR, test_df, TEST_CONTRACT, FINAL_CONTRACT = load_unlocked_context()
    audio_paths, audio_integrity_sha = ensure_frozen_audio(
        test_df, STATE_DIR, parquet_revision=TEST_CONTRACT["parquet_revision"],
    )
    expected_uids = test_df["record_uid"].astype(str).tolist()

    def _release_cuda():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    t0 = time.time()
    seq = run_cascaded_c0_peak_safe(
        restore_asr=lambda: restore_asr_for_rq1(
            state_dir=ASR_STATE_DIR, local_ckpt_root=RUNTIME.local_asr_ckpt_root, device=DEVICE,
        ),
        restore_mt=lambda: restore_mt_for_rq1(
            state_dir=MT_STATE_DIR, local_ckpt_root=RUNTIME.local_mt_ckpt_root, device=DEVICE,
        ),
        predict_asr=lambda asr: run_asr_predictions(
            model=asr["model"], processor=asr["processor"], frame=test_df, audio_paths=audio_paths, batch_size=1,
        ),
        predict_mt=lambda mt, asr_rows: run_mt_from_asr(
            model=mt["model"], tokenizer=mt["tokenizer"], asr_predictions=asr_rows,
            max_source_length=int(mt["training_contract"]["max_source_length"]),
            generation_max_length=int(mt["training_contract"]["generation_max_length"]),
            num_beams=int(mt["training_contract"]["num_beams"]),
            batch_size=int(mt["training_contract"].get("per_device_eval_batch_size") or 8),
        ),
        assert_asr_proof=lambda asr: assert_restored_checkpoint_matches_proof(
            asr, FINAL_CONTRACT["checkpoint_proof"]["asr"], system="asr",
        ),
        assert_mt_proof=lambda mt: assert_restored_checkpoint_matches_proof(
            mt, FINAL_CONTRACT["checkpoint_proof"]["mt"], system="mt",
        ),
        expected_uids=expected_uids,
        asr_allowed_root=RUNTIME.local_asr_ckpt_root,
        mt_allowed_root=RUNTIME.local_mt_ckpt_root,
        cleanup_local=cleanup_local_rq1_experiment,
        release_cuda=_release_cuda,
    )
    total = time.time() - t0
    asr_rows = seq["asr_rows"]
    c0 = seq["c0"]
    trunc = dict(getattr(c0, "attrs", {}).get("mt_truncation_audit") or {})
    c0_path = STATE_DIR / "c0_predictions.csv"
    atomic_write_csv(c0, c0_path)
    payload = {
        "status": STATUS_RQ1_C0,
        "rq1_final_contract_hash": FINAL_CONTRACT["rq1_final_contract_hash"],
        "prediction_sha256": sha256_file(c0_path),
        "audio_integrity_sha256": audio_integrity_sha,
        "gpu_name": torch.cuda.get_device_name(0),
        "n": len(c0),
        "total_seconds": total,
        "asr_best_checkpoint": seq["asr_best_checkpoint"],
        "mt_best_checkpoint": seq["mt_best_checkpoint"],
        "mt_truncation_audit": trunc,
        "restore_order": seq["restore_order"],
        "asr_local_cleaned": not Path(seq["asr_local_experiment_dir"]).exists(),
        "mt_local_cleaned": not Path(seq["mt_local_experiment_dir"]).exists(),
    }
    if not payload["asr_local_cleaned"] or not payload["mt_local_cleaned"]:
        raise RuntimeError("C0 local restore trees were not cleaned from disk")
    atomic_write_json(STATE_DIR / "c0_summary.json", payload)
    del asr_rows, c0, seq
    _release_cuda()
    print(json.dumps({k: payload[k] for k in (
        "status", "n", "audio_integrity_sha256", "mt_truncation_audit", "restore_order",
        "asr_local_cleaned", "mt_local_cleaned",
    )}, indent=2))


In [ ]:
# Cell 8 — Stage: run_direct (D0), exception-safe GPU+disk peak control
if RQ1_STAGE == "run_direct":
    if DEVICE != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("run_direct requires CUDA")
    STATE_DIR, test_df, TEST_CONTRACT, FINAL_CONTRACT = load_unlocked_context()
    audio_paths, audio_integrity_sha = ensure_frozen_audio(
        test_df, STATE_DIR, parquet_revision=TEST_CONTRACT["parquet_revision"],
    )
    expected_uids = test_df["record_uid"].astype(str).tolist()

    def _release_cuda():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    t0 = time.time()
    seq = run_direct_d0_peak_safe(
        restore_direct=lambda: restore_direct_for_rq1(
            state_dir=DIRECT_STATE_DIR, local_ckpt_root=RUNTIME.local_direct_ckpt_root, device=DEVICE,
        ),
        predict_direct=lambda direct: run_direct_predictions(
            model=direct["model"], feature_extractor=direct["feature_extractor"], tokenizer=direct["tokenizer"],
            frame=test_df, audio_paths=audio_paths,
            target_lang=str(direct["training_contract"]["target_lang"]),
            generation_max_length=int(direct["training_contract"]["generation_max_length"]),
            num_beams=int(direct["training_contract"]["num_beams"]),
            batch_size=int(direct["training_contract"].get("per_device_eval_batch_size") or 1),
        ),
        assert_direct_proof=lambda direct: assert_restored_checkpoint_matches_proof(
            direct, FINAL_CONTRACT["checkpoint_proof"]["direct"], system="direct",
        ),
        expected_uids=expected_uids,
        direct_allowed_root=RUNTIME.local_direct_ckpt_root,
        cleanup_local=cleanup_local_rq1_experiment,
        release_cuda=_release_cuda,
    )
    runtime = time.time() - t0
    d0 = seq["d0"]
    d0_path = STATE_DIR / "d0_predictions.csv"
    atomic_write_csv(d0, d0_path)
    payload = {
        "status": STATUS_RQ1_D0,
        "rq1_final_contract_hash": FINAL_CONTRACT["rq1_final_contract_hash"],
        "prediction_sha256": sha256_file(d0_path),
        "audio_integrity_sha256": audio_integrity_sha,
        "gpu_name": torch.cuda.get_device_name(0),
        "n": len(d0),
        "total_seconds": runtime,
        "direct_best_checkpoint": seq["direct_best_checkpoint"],
        "restore_order": seq["restore_order"],
        "direct_local_cleaned": not Path(seq["direct_local_experiment_dir"]).exists(),
    }
    if not payload["direct_local_cleaned"]:
        raise RuntimeError("D0 local restore tree was not cleaned from disk")
    atomic_write_json(STATE_DIR / "d0_summary.json", payload)
    del d0, seq
    _release_cuda()
    print(json.dumps({k: payload[k] for k in ("status", "n", "audio_integrity_sha256", "direct_local_cleaned", "restore_order")}, indent=2))


In [ ]:
# Cell 9 — Stage: finalize paired metrics, cluster bootstrap, slices, errors, hashes
if RQ1_STAGE == "finalize":
    STATE_DIR, test_df, TEST_CONTRACT, FINAL_CONTRACT = load_unlocked_context()
    c0_path = STATE_DIR / "c0_predictions.csv"; d0_path = STATE_DIR / "d0_predictions.csv"
    c0_sum_path = STATE_DIR / "c0_summary.json"; d0_sum_path = STATE_DIR / "d0_summary.json"
    if not all(p.is_file() for p in (c0_path, d0_path, c0_sum_path, d0_sum_path)):
        raise RuntimeError("Missing C0/D0 prediction artifacts; run_cascaded and run_direct first")
    c0 = pd.read_csv(c0_path); d0 = pd.read_csv(d0_path)
    c0_sum = json.loads(c0_sum_path.read_text(encoding="utf-8"))
    d0_sum = json.loads(d0_sum_path.read_text(encoding="utf-8"))
    assert_stage_summary_matches_final_contract(
        c0_sum, final_contract=FINAL_CONTRACT, test_contract=TEST_CONTRACT,
        expected_status=STATUS_RQ1_C0, label="C0",
    )
    assert_stage_summary_matches_final_contract(
        d0_sum, final_contract=FINAL_CONTRACT, test_contract=TEST_CONTRACT,
        expected_status=STATUS_RQ1_D0, label="D0",
    )
    if sha256_file(c0_path) != c0_sum.get("prediction_sha256"):
        raise RuntimeError("C0 predictions mutated after run_cascaded")
    if sha256_file(d0_path) != d0_sum.get("prediction_sha256"):
        raise RuntimeError("D0 predictions mutated after run_direct")
    integrity_path = STATE_DIR / "rq1_audio_integrity.csv"
    if not integrity_path.is_file():
        raise RuntimeError("Missing rq1_audio_integrity.csv for finalize")
    audio_integrity_sha = recompute_and_assert_finalize_integrity(
        integrity_path=integrity_path, c0_summary=c0_sum, d0_summary=d0_sum,
    )
    paired = join_paired_predictions(test_df, c0, d0)
    metrics = system_metrics(paired)
    bootstrap = paired_bootstrap(
        paired,
        n_samples=int(FINAL_CONTRACT["bootstrap_samples"]),
        seed=int(FINAL_CONTRACT["seed"]),
        confidence=float(FINAL_CONTRACT["confidence"]),
        cluster_col=str(FINAL_CONTRACT["cluster_col"]),
    )
    bootstrap_ok = True
    try:
        assert_rq1_bootstrap_gate(
            bootstrap, final_contract=FINAL_CONTRACT, test_contract=TEST_CONTRACT,
        )
    except RuntimeError:
        bootstrap_ok = False
    slices = slice_metrics(paired)
    errors = error_examples(paired, top_k=20)
    atomic_write_csv(paired, STATE_DIR / "paired_predictions.csv")
    atomic_write_csv(slices, STATE_DIR / "rq1_slice_metrics.csv")
    atomic_write_csv(errors, STATE_DIR / "rq1_error_examples.csv")
    atomic_write_json(STATE_DIR / "rq1_metrics.json", metrics)
    atomic_write_json(STATE_DIR / "rq1_bootstrap.json", bootstrap)
    paths = {
        "paired_predictions.csv": STATE_DIR / "paired_predictions.csv",
        "rq1_metrics.json": STATE_DIR / "rq1_metrics.json",
        "rq1_bootstrap.json": STATE_DIR / "rq1_bootstrap.json",
        "rq1_slice_metrics.csv": STATE_DIR / "rq1_slice_metrics.csv",
        "rq1_error_examples.csv": STATE_DIR / "rq1_error_examples.csv",
        "c0_predictions.csv": c0_path,
        "d0_predictions.csv": d0_path,
        "rq1_audio_integrity.csv": integrity_path,
        "rq1_test_contract.json": STATE_DIR / "rq1_test_contract.json",
        "rq1_final_contract.json": STATE_DIR / "rq1_final_contract.json",
    }
    manifest = prediction_artifact_manifest(paths)
    atomic_write_json(STATE_DIR / "rq1_artifact_manifest.json", manifest)
    verdict = derive_final_status(
        test_contract_ok=True,
        c0_ok=c0_sum.get("status") == STATUS_RQ1_C0,
        d0_ok=d0_sum.get("status") == STATUS_RQ1_D0,
        paired_uid_order_ok=paired["record_uid"].astype(str).tolist() == test_df["record_uid"].astype(str).tolist(),
        references_identical=True,
        metrics_finite=finite_metric_dict(metrics["c0"], ("sacrebleu", "chrfpp")) and finite_metric_dict(metrics["d0"], ("sacrebleu", "chrfpp")),
        artifacts_hashed=bool(manifest.get("manifest_hash")),
        bootstrap_ok=bootstrap_ok,
    )
    final = {
        "status": verdict["status"],
        "checks": verdict["checks"],
        "failed_checks": verdict["failed_checks"],
        "rq1_final_contract_hash": FINAL_CONTRACT["rq1_final_contract_hash"],
        "rq1_test_contract_hash": TEST_CONTRACT["rq1_test_contract_hash"],
        "metrics": metrics,
        "bootstrap": bootstrap,
        "audio_integrity_sha256": audio_integrity_sha,
        "artifact_manifest_sha256": sha256_file(STATE_DIR / "rq1_artifact_manifest.json"),
        "ready_rq1_final": verdict["status"] == STATUS_RQ1_FINAL,
        "n_test": int(len(test_df)),
        "n_clusters": int(TEST_CONTRACT.get("n_clusters") or 0),
    }
    atomic_write_json(STATE_DIR / "rq1_final_summary.json", final)
    print(json.dumps(final, indent=2))


## Interpretation boundary

Notebook 06 outputs paired evidence for RQ1. It does **not** retune NB03/NB04/NB05 after the frozen test has been unlocked. Any thesis interpretation must preserve the exact test contract and artifact hashes written here.
